<h1><center>Recommender Systems YSDA Course</center></h1>

## **Classic algorithms & Ranking** practice

Today we'll have a look at some good baselines for candidate generation, features and ranking algorithms for RecSys.

### Imports and setup

In [1]:
# If you are running this notebook in Colab or similar, clone the repo and install deps first:
# !pip install polars scipy scikit-learn duckdb catboost tqdm

import random

import numpy as np
import polars as pl
import scipy as sp
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split

random.seed(42)
np.random.seed(42)

### Data: Lavka (actions + products)

In [4]:
# download link - https://disk.yandex.ru/d/KvyEFHD-sjHSAg

DATA_DIR = "~/Downloads"
ACTIONS_PATH = f"{DATA_DIR}/dm_action.parquet"
PRODUCTS_PATH = f"{DATA_DIR}/dm_product.parquet"

actions = pl.read_parquet(ACTIONS_PATH)
products = pl.read_parquet(PRODUCTS_PATH)

interactions = (
    actions
    .filter(pl.col("action_type") == "AT_CartUpdate")
    .select(
        pl.col("user_id"),
        pl.col("item_id"),
        pl.col("timestamp"),
    )
    .with_columns(pl.lit(1).alias("rating"))
    .sort("timestamp")
)

# Global time split
split_idx = int(len(interactions) * 0.8)
train = interactions[:split_idx]
test = interactions[split_idx:]

# Keep only users that exist in train
train_users = train.select(pl.col("user_id").unique())
test = test.join(train_users, on="user_id", how="inner")

train_by_user = (
    train
    .group_by("user_id")
    .agg(pl.col("item_id").unique().alias("items"))
)

test_by_user = (
    test
    .group_by("user_id")
    .agg(pl.col("item_id").unique().alias("items"))
)

train_by_user = {row["user_id"]: set(row["items"]) for row in train_by_user.iter_rows(named=True)}
test_by_user = {row["user_id"]: set(row["items"]) for row in test_by_user.iter_rows(named=True)}

In [5]:
train

user_id,item_id,timestamp,rating
u64,u64,i64,i32
17527326229946774021,12483267338427566363,1672533969,1
17527326229946774021,3842715519498747470,1672533982,1
17527326229946774021,18170588393614160842,1672533989,1
17527326229946774021,4391631274737042024,1672534052,1
17527326229946774021,17143610210879625024,1672534095,1
…,…,…,…
14740182241486445575,8697267488153818286,1701076130,1
13746347995296249849,2690757275610358522,1701076812,1
13746347995296249849,17387805830813612125,1701076846,1


### Metrics and evaluation

In [11]:
def recall_at_k(preds: list[int], positives: set[int], k: int) -> float:
    if not positives:
        return 0.0
    preds = preds[:k]
    return len(set(preds) & positives) / len(positives)


def ndcg_at_k(preds: list[int], positives: set[int], k: int) -> float:
    if not positives:
        return 0.0
    dcg = 0.0
    for rank, item_id in enumerate(preds[:k], start=1):
        if item_id in positives:
            dcg += 1.0 / np.log2(rank + 1)

    ideal_hits = min(len(positives), k)
    idcg = sum(1.0 / np.log2(rank + 1) for rank in range(1, ideal_hits + 1))
    return dcg / idcg if idcg > 0 else 0.0


def evaluate_recommender(recommend_fn, test_by_user: dict[int, set[int]], k_list=(10, 50)):
    max_k = max(k_list)
    metrics = {f"recall@{k}": [] for k in k_list}
    metrics.update({f"ndcg@{k}": [] for k in k_list})

    for user_id, positives in tqdm(test_by_user.items(), total=len(test_by_user)):
        preds = recommend_fn(user_id, max_k)
        for k in k_list:
            metrics[f"recall@{k}"].append(recall_at_k(preds, positives, k))
            metrics[f"ndcg@{k}"].append(ndcg_at_k(preds, positives, k))

    return {name: float(np.mean(values)) for name, values in metrics.items()}


## Classic algos for CG / feature extraction (or ranking)

In [8]:
def build_interaction_matrix(ratings: pl.DataFrame, additive: bool = False):
    users = ratings.select(pl.col("user_id").unique())
    items = ratings.select(pl.col("item_id").unique())
    num_users = len(users)
    num_items = len(items)

    user_id2idx = {row["user_id"]: i for i, row in enumerate(users.iter_rows(named=True))}
    item_id2idx = {row["item_id"]: i for i, row in enumerate(items.iter_rows(named=True))}
    user_idx2id = {v: k for k, v in user_id2idx.items()}
    item_idx2id = {v: k for k, v in item_id2idx.items()}

    R = sp.sparse.lil_array((num_users, num_items))
    for row in ratings.iter_rows(named=True):
        user_idx = user_id2idx[row["user_id"]]
        item_idx = item_id2idx[row["item_id"]]
        if additive:
            R[user_idx, item_idx] += row["rating"]
        else:
            R[user_idx, item_idx] = row["rating"]

    return R.tocsr(), (user_id2idx, item_id2idx, user_idx2id, item_idx2id)


### EASE (Embarrassingly Shallow AutoEncoder)

This is the model from [Embarrassingly Shallow Autoencoders for Sparse Data](https://arxiv.org/pdf/1905.03375).

From practice we know that we can present our training data as a sparse matrix $X \in \mathbb{R}^{|U| x |I|}$, where each element $x_{ui}$ is a binary value, which is positive if the user $u$ had an interaction with item $i$, or rating from their interactions, or number of interactions.

EASE suggests a linear model, that encodes all items as an item-item matrix $B$, and each item can be represented as a linear combination of other items.
We consider two features. The first is categorical — the ID of the item being evaluated. The second group consists of items with which the user has had positive interactions (excluding the item in question). We then take the cross-product of these two groups to generate features like: [currently evaluating item i, item j appears in the user’s history]. We train a linear model on these cross-features. The target is 1 for positive interactions and 0 otherwise (even if the user hasn’t been shown the item).

We use MSE loss and L2 regularization (a convex loss function), adding the constraint that diagonal weights must be zero. This constraint, $diag(B) = 0$, is crucial as to avoid the trivial solution $B = I$ (self-similarity of items), where $I$ is the identity matrix.

The optimization problem in matrix form is as follows:

$$\min_{B} ||X - XB||^2_F + \lambda ||B||^2_F$$
$$\text{s.t. diag}(B) = 0$$

Turns out, the problem has an exact analytical solution! And can be solved with both sparse and dense matrices.

In [9]:
class EASE:
    def __init__(self, l2_reg: float = 1000.0):
        self.l2_reg = l2_reg

    def fit(self, interactions: pl.DataFrame):
        self.R, (self.user_id2idx, self.item_id2idx, self.user_idx2id, self.item_idx2id) = (
            build_interaction_matrix(interactions)
        )
        G = (self.R.T @ self.R).toarray()
        G[np.diag_indices_from(G)] += self.l2_reg
        P = np.linalg.inv(G)
        B = -P / np.diag(P)
        np.fill_diagonal(B, 0.0)
        self.B = B

    def recommend(self, user_id: int, n: int = 10) -> list[int]:
        user_idx = self.user_id2idx.get(user_id)
        if user_idx is None:
            return []
        user_row = self.R[user_idx]
        scores = (user_row @ self.B)
        top_idx = np.argsort(-scores)[:n]
        return [self.item_idx2id[i] for i in top_idx]

In [12]:
ease = EASE(l2_reg=0.1)
ease.fit(train)
metrics = evaluate_recommender(
    lambda user_id, n: ease.recommend(user_id, n),
    test_by_user,
    k_list=(10, 50),
)
print(metrics)

  0%|          | 0/901 [00:00<?, ?it/s]

{'recall@10': 0.05955205800699852, 'recall@50': 0.16891429032644478, 'ndcg@10': 0.14602317662502196, 'ndcg@50': 0.15236997532798108}


### SLIM (Sparse Linear Methods)

The next model in our arsenal will be **S**parse **Li**near **M**ethod a.k.a. [SLIM](https://ieeexplore.ieee.org/document/6137254) (PDF is available [here](https://www.researchgate.net/profile/George_Karypis/publication/220765374_SLIM_Sparse_Linear_Methods_for_Top-N_Recommender_Systems/links/549ee9ac0cf257a635fe7010.pdf)).

Actually, SLIM was introduced long before EASE, and EASE cites SLIM (and uses a lot of it as a foundation), not the other way around. For educational purposes, we switched their order of presentation in the homework. The algorithm solves the same problem of finding best item-item weight matrix, but with different loss - the main difference is L1 regularization for sparsity component, which helps the algorithm to build a sparse weight matrix, which is both memory-efficient and compute-efficient, if you organize your storage accordingly.


The optimization problem:

$$\min_{B} \frac{1}{2}||X - XB||^2_F + \frac{\beta}{2} ||B||^2_F + \lambda ||B||_F$$
$$\text{subject to } B \geq 0 \text{, diag}(B) = 0$$

This is solvable with coordinate descent by projecting each individual weight (non-negative constraint) and manually setting diagonal weights to zero during all the steps. Moreover, each item weight vector is independent from others, so the task is easily parallelizable across items. However, there aren't any optimizers that can do that out of the box. As proposed in this [paper](https://www.slideshare.net/slideshow/efficient-slides/27138952), we'll simplify the problem a bit by dropping the $diag(B) = 0$ constraint and setting the item column of the interaction matrix to zero when fitting for it. Let's see, whether it's still a viable model - simple scikit-learn should do the trick.

We'll implement SLIM model using sklearn ElasticNet solver and optionally multiprocessing. The steps are as follows:
- Build sparse matrix of interactions
- For each item $u$:
    - Get interaction matrix $X$ - the features
    - Extract target vector $y = X_u \in R^{|U|}$ - all interactions with this item
    - Find solution and save it
- Extract sparse coefs to one weight matrix B

In [13]:
%%writefile slim.py

import multiprocessing as mp
from functools import partial
import polars as pl
import numpy as np
import scipy as sp
import sklearn.linear_model


def _solve_item(idx, R, alpha, l1_ratio, positive):
    y = R[:, idx].todense()
    solver = sklearn.linear_model.ElasticNet(
        alpha=alpha,
        l1_ratio=l1_ratio,
        positive=positive,
        copy_X=False,
        fit_intercept=False,
        precompute=True,
        selection="random",
        max_iter=100,
    )
    solver.fit(R, y)
    return solver.sparse_coef_


class SLIM:
    def __init__(self,
                 alpha: float = 0.1,
                 l1_ratio: float = 0.01,
                 positive: bool = False,
                 num_processes: int | None = None,
                 ):
        self.l1_ratio = l1_ratio
        self.alpha = alpha
        self.positive = positive
        self.num_processes = num_processes

    def fit(self, interactions: pl.DataFrame):
        self.R, (self.user_id2idx, self.item_id2idx, self.user_idx2id, self.item_idx2id) = (
            build_interaction_matrix(interactions)
        )
        self.num_items = self.R.shape[1]
        self._solve = partial(
            _solve_item,
            R=self.R,
            alpha=self.alpha,
            l1_ratio=self.l1_ratio,
            positive=self.positive
        )
        if self.num_processes == 1:
            self.results = [
                self._solve(i) for i in range(self.num_items)
            ]
        else:
            pool = mp.Pool(self.num_processes)
            self.results = pool.map(self._solve, range(self.num_items))
        self.B = sp.sparse.vstack(self.results).T

    def recommend(self, user_id: int, k: int):
        user_idx = self.user_id2idx.get(user_id)
        if user_idx is None:
            return []
        item_scores = (self.R[user_idx, :] @ self.B).todense()
        top_items = np.argsort(item_scores)[::-1][:k]
        results = [self.item_idx2id[item_idx] for item_idx in top_items]
        return results


def build_interaction_matrix(ratings: pl.DataFrame, additive: bool = False):
    users = ratings.select(pl.col("user_id").unique())
    items = ratings.select(pl.col("item_id").unique())
    num_users = len(users)
    num_items = len(items)

    user_id2idx = {row["user_id"]: i for i, row in enumerate(users.iter_rows(named=True))}
    item_id2idx = {row["item_id"]: i for i, row in enumerate(items.iter_rows(named=True))}
    user_idx2id = {v: k for k, v in user_id2idx.items()}
    item_idx2id = {v: k for k, v in item_id2idx.items()}

    R = sp.sparse.lil_array((num_users, num_items))
    for row in ratings.iter_rows(named=True):
        user_idx = user_id2idx[row["user_id"]]
        item_idx = item_id2idx[row["item_id"]]
        if additive:
            R[user_idx, item_idx] += row["rating"]
        else:
            R[user_idx, item_idx] = row["rating"]

    return R.tocsr(), (user_id2idx, item_id2idx, user_idx2id, item_idx2id)

Overwriting slim.py


In [15]:
!uv run python train_slim.py --path ~/Downloads/dm_action.parquet

100%|████████████████████████████████████████| 901/901 [00:02<00:00, 427.28it/s]
{'recall@10': 0.06182292510626886, 'recall@50': 0.17671654015030056, 'ndcg@10': 0.15609016363950834, 'ndcg@50': 0.1610166521457917}


### Logistic Matrix Factorization

Logistic Matrix Factorization (LMF) is a probabilistic MF model for implicit feedback. Instead of fitting ratings directly, we model the probability of an interaction with a logistic link:

$$p(r_{ui}=1) = \sigma(u_u^T v_i + b_u + b_i)$$

Here $u_u$ and $v_i$ are user and item embeddings, and $b_u, b_i$ are biases. The loss is the negative log-likelihood of observed positives vs. negatives (usually sampled):

$$\min_{U,V,b} \sum_{(u,i) \in \mathcal{P}} -\log \sigma(x_{ui}) + \sum_{(u,j) \in \mathcal{N}} -\log (1-\sigma(x_{uj})) + \lambda (||U||^2 + ||V||^2)$$

where $x_{ui} = u_u^T v_i + b_u + b_i$, $\mathcal{P}$ are positive interactions, and $\mathcal{N}$ are sampled negatives. We optimize this with gradient descent by first doing a user optimization, then item optimization; other possible options are SGD with randonmly chosen batches of user-items pairs.


In [16]:
class LogisticMF:
    def __init__(
        self,
        dim: int = 128,
        max_iter: int = 50,
        lr: float = 0.003,
        reg_embeddings: float = 0.01,
        reg_biases: float = 0.01,
        neg_samples: int = 5,
        seed: int = 42,
    ):
        self.lr = lr
        self.dim = dim
        self.max_iter = max_iter
        self.reg_embeddings = reg_embeddings
        self.reg_biases = reg_biases
        self.neg_samples = neg_samples
        self.seed = seed

    def _init_parameters(self, ratings: pl.DataFrame):
        self.R, (
            self.user_id2idx,
            self.item_id2idx,
            self.user_idx2id,
            self.item_idx2id,
        ) = build_interaction_matrix(ratings)
        self.n_users, self.n_items = self.R.shape
        self.user_vectors = np.random.normal(size=(self.n_users, self.dim))
        self.item_vectors = np.random.normal(size=(self.n_items, self.dim))
        self.user_biases = np.zeros((self.n_users, 1))
        self.item_biases = np.zeros((self.n_items, 1))

    def fit(self, ratings: pl.DataFrame):
        self._init_parameters(ratings)
        for _ in tqdm(range(self.max_iter)):
            error = self.R - sp.special.expit(
                self.user_vectors @ self.item_vectors.T
                + self.user_biases
                + self.item_biases.T
            )
            vector = error @ self.item_vectors - self.reg_embeddings * self.user_vectors
            bias = error.sum(axis=1)[:, None] - self.reg_biases * self.user_biases
            self.user_vectors += self.lr * vector
            self.user_biases += self.lr * bias
            
            error = self.R - sp.special.expit(
                self.user_vectors @ self.item_vectors.T
                + self.user_biases
                + self.item_biases.T
            )
            vector = error.T @ self.user_vectors - self.reg_embeddings * self.item_vectors
            bias = error.sum(axis=0)[:, None] - self.reg_biases * self.item_biases
            self.item_vectors += self.lr * vector
            self.item_biases += self.lr * bias
        return self
    
    def recommend(self, user_id: int, n: int):
        user_idx = self.user_id2idx.get(user_id)
        if user_idx is None:
            return []
        user_vec = self.user_vectors[user_idx]
        user_bias = self.user_biases[user_idx]
        scores = (
            user_vec @ self.item_vectors.T # 1 x n_items
            + self.item_biases.T # 1 x n_items
            + user_bias # 1 x 1
        ).flatten()
        top_idx = np.argsort(-scores)[:n]
        return [self.item_idx2id[i] for i in top_idx]

In [17]:
model = LogisticMF()
model.fit(train)
metrics = evaluate_recommender(
    lambda user_id, n: model.recommend(user_id, n),
    test_by_user,
    k_list=(10, 50),
)
metrics

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/901 [00:00<?, ?it/s]

{'recall@10': 0.05282443033266831,
 'recall@50': 0.1070687989173163,
 'ndcg@10': 0.11581120551442872,
 'ndcg@50': 0.10322194392583671}

## Ranking with CatBoost

We build a re-ranking dataset from user requests. This section follows the Week 4 practice with minimal changes,
except that we use the Lavka files from Downloads and remove the Reranker/Softmax sampler parts.


In [18]:
import duckdb
from catboost import CatBoostRanker, CatBoostClassifier, Pool

In [19]:
FILL_VALUE = -999999999.0
DATA_DIR = "/Users/rmnigm/Downloads"
ACTIONS_PATH = f"{DATA_DIR}/dm_action.parquet"
PRODUCTS_PATH = f"{DATA_DIR}/dm_product.parquet"

# Build a features table with DuckDB (saved locally)
sql = f'''
WITH
    actions as (
        SELECT
            a.user_id,
            a.source_type,
            p.item_category as product_category,
            a.item_id,
            a.request_id,
            a.action_type,
            a.position_in_request,
            make_timestamp(a.timestamp * 1000000) as timestamp,
            date_trunc('day', make_timestamp(a.timestamp * 1000000)) as day
        FROM '{ACTIONS_PATH}' a
        LEFT JOIN '{PRODUCTS_PATH}' p
        ON a.item_id = p.item_id
        WHERE a.action_type IN ('AT_CartUpdate', 'AT_View', 'AT_Click')
    ),
    user_agg as (
        SELECT
            user_id,
            sum(case when action_type = 'AT_CartUpdate' then 1 else 0 end) as user_cart_updates,
            sum(case when action_type = 'AT_View' then 1 else 0 end) as user_views,
            sum(case when action_type = 'AT_Click' then 1 else 0 end) as user_clicks
        FROM actions
        GROUP BY user_id
    ),
    item_agg as (
        SELECT
            item_id,
            sum(case when action_type = 'AT_CartUpdate' then 1 else 0 end) as item_cart_updates,
            sum(case when action_type = 'AT_View' then 1 else 0 end) as item_views,
            sum(case when action_type = 'AT_Click' then 1 else 0 end) as item_clicks
        FROM actions
        GROUP BY item_id
    ),
    cart_updates as (
        SELECT
            user_id,
            item_id,
            timestamp,
            date_diff('second', lag(timestamp) over (partition by user_id, item_id order by timestamp), timestamp) as delta_seconds
        FROM actions
        WHERE action_type = 'AT_CartUpdate'
    ),
    u2i as (
        SELECT
            user_id,
            item_id,
            count(*) as u2i_cart_updates,
            avg(delta_seconds) as u2i_mean_time_between_cartupdates
        FROM cart_updates
        GROUP BY user_id, item_id
    )
SELECT
    a.*,
    (user_cart_updates / nullif(user_cart_updates + user_views + user_clicks, 0)) as user_cart_update_turn_rate,
    (user_clicks / nullif(user_cart_updates + user_views + user_clicks, 0)) as user_click_turn_rate,
    (user_cart_updates / nullif(user_views + user_clicks, 0)) as user_conversion_rate,

    (item_cart_updates / nullif(item_cart_updates + item_views + item_clicks, 0)) as item_cart_update_turn_rate,
    (item_clicks / nullif(item_cart_updates + item_views + item_clicks, 0)) as item_click_turn_rate,
    (item_cart_updates / nullif(item_views + item_clicks, 0)) as item_conversion_rate,

    u2i.u2i_cart_updates,
    u2i.u2i_mean_time_between_cartupdates
FROM actions a
LEFT JOIN user_agg u on a.user_id = u.user_id
LEFT JOIN item_agg i on a.item_id = i.item_id
LEFT JOIN u2i on a.user_id = u2i.user_id and a.item_id = u2i.item_id
'''

duckdb.sql(sql).write_parquet("train_with_features.parquet")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [20]:
dataset = pl.read_parquet("train_with_features.parquet")

NUM_FEATURE_COLUMNS = [
    "user_cart_update_turn_rate",
    "user_click_turn_rate",
    "user_conversion_rate",
    "item_cart_update_turn_rate",
    "item_click_turn_rate",
    "item_conversion_rate",
    "u2i_cart_updates",
    "u2i_mean_time_between_cartupdates",
]

CAT_FEATURE_COLUMNS = [
    "product_category",
]

FEATURE_COLUMNS = NUM_FEATURE_COLUMNS + CAT_FEATURE_COLUMNS

In [21]:
# Fill missing numeric features and categories
dataset = dataset.with_columns([pl.col(c).fill_null(FILL_VALUE) for c in NUM_FEATURE_COLUMNS])
dataset = dataset.with_columns([pl.col(c).fill_null('unknown') for c in CAT_FEATURE_COLUMNS])

In [ ]:
def build_target(action_type: str):
    if action_type == "AT_CartUpdate":
        return 1.0
    elif action_type == "AT_View":
        return 0.0

cbm_dataset = (
    dataset
    .sort("timestamp")
    .filter(pl.col("request_id").is_not_null())
    .filter(pl.col("action_type").is_in(["AT_CartUpdate", "AT_View"]))
    .with_columns(
        pl.col("request_id").cast(str).alias("group_id"),
        pl.col("action_type").map_elements(build_target, return_dtype=float).alias("target")
    )
    .with_columns(
        target=pl.col('target').max().over(partition_by=[pl.col('group_id'), pl.col('item_id')])
    )
    .unique()
)

train_rank, test_rank = train_test_split(cbm_dataset, test_size=0.2, shuffle=False)

train_rank = train_rank.sort("group_id")
test_rank = test_rank.sort("group_id")

In [25]:
train_pool = Pool(
    train_rank.select(FEATURE_COLUMNS).to_numpy(),
    feature_names=FEATURE_COLUMNS,
    cat_features=CAT_FEATURE_COLUMNS,
    label=train_rank["target"].to_numpy(),
    group_id=train_rank["group_id"].to_numpy()
)

test_pool = Pool(
    test_rank.select(FEATURE_COLUMNS).to_numpy(),
    feature_names=FEATURE_COLUMNS,
    cat_features=CAT_FEATURE_COLUMNS,
    label=test_rank["target"].to_numpy(),
    group_id=test_rank["group_id"].to_numpy()
)


In [26]:
model = CatBoostClassifier(iterations=100, eval_metric="NDCG:top=10")
model.fit(train_pool, eval_set=test_pool, early_stopping_rounds=20, plot=True, verbose=False)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

In [27]:
pl.DataFrame({
    "importance": model.feature_importances_,
    "feature": model.feature_names_
}).sort("importance", descending=True)

importance,feature
f64,str
89.78733,"""u2i_cart_updates"""
4.753105,"""user_click_turn_rate"""
2.037254,"""user_cart_update_turn_rate"""
1.844726,"""user_conversion_rate"""
0.487935,"""u2i_mean_time_between_cartupda…"
0.443402,"""item_click_turn_rate"""
0.224732,"""item_conversion_rate"""
0.217972,"""item_cart_update_turn_rate"""
0.203545,"""product_category"""


In [28]:
metrics = model.eval_metrics(test_pool, ["AUC:type=Ranking", "NDCG", "NDCG:top=10"])
for metric in metrics:
    print(metric)
    print(np.mean(metrics[metric]))

AUC:type=Ranking
0.9347249724906624
NDCG:type=Base
0.9838861801417976
NDCG:top=10;type=Base
0.9835527106775871


### Ranking tricks: undersampling

In [29]:
(
    dataset
    .group_by("request_id")
    .agg(
        (pl.col("action_type") == "AT_CartUpdate").sum().alias("cart_updates"),
        (pl.col("action_type") == "AT_View").sum().alias("views"),
        (pl.col("action_type") == "AT_Click").sum().alias("clicks"),
    )
    .select(
        pl.col("cart_updates").mean().alias("cart_updates_mean"),
        pl.col("views").mean().alias("views_mean"),
        pl.col("clicks").mean().alias("clicks_mean"),
    )
)

cart_updates_mean,views_mean,clicks_mean
f64,f64,f64
0.461508,23.577307,0.376858


In [30]:
resampled_dataset = dataset.with_row_index()

request_views = (
    resampled_dataset
    .filter(pl.col("action_type") == "AT_View")
    .select(
        pl.col("index").sort_by("position_in_request").head(10).over("request_id", mapping_strategy="explode"),
        pl.lit(True).alias("view_is_ok"),
    )
)

request_clicks = (
    resampled_dataset
    .filter(pl.col("action_type") == "AT_Click")
    .select(
        pl.col("index").sort_by("position_in_request").head(3).over("request_id", mapping_strategy="explode"),
        pl.lit(True).alias("click_is_ok"),
    )
)

requests_with_cartupdate = (
    resampled_dataset
    .filter(pl.col("action_type") == "AT_CartUpdate")
    .select("request_id").unique()
)


In [31]:
resampled_dataset = (
    resampled_dataset
    .filter(pl.col("request_id").is_not_null())
    .join(requests_with_cartupdate, on="request_id", how="inner")
    .join(request_views, on="index", how="left")
    .join(request_clicks, on="index", how="left")
    .filter((pl.col("action_type") == "AT_CartUpdate") | pl.col("view_is_ok") | pl.col("click_is_ok"))
)

In [36]:
resampled_cbm_dataset = (
    resampled_dataset
    .filter(pl.col("action_type").is_in(["AT_CartUpdate", "AT_View"]))
    .with_columns(
        pl.col("request_id").cast(str).alias("group_id"),
        pl.struct("action_type").map_elements(lambda x: build_target(x["action_type"]), return_dtype=float).alias("target")
    )
    .with_columns(
        target=pl.col('target').max().over(partition_by=[pl.col('group_id'), pl.col('item_id')])
    )
    .unique()
    .sort("timestamp")
)

resampled_train, resampled_test = train_test_split(resampled_cbm_dataset, test_size=0.2, shuffle=False)
resampled_train = resampled_train.sort("group_id")
resampled_test = resampled_test.sort("group_id")

In [43]:
resampled_train_pool = Pool(
    resampled_train.select(FEATURE_COLUMNS).to_numpy(),
    feature_names=FEATURE_COLUMNS,
    cat_features=CAT_FEATURE_COLUMNS,
    label=resampled_train["target"].to_numpy(),
    group_id=resampled_train["group_id"].to_numpy()
)

resampled_test_pool = Pool(
    resampled_test.select(FEATURE_COLUMNS).to_numpy(),
    feature_names=FEATURE_COLUMNS,
    cat_features=CAT_FEATURE_COLUMNS,
    label=resampled_test["target"].to_numpy(),
    group_id=resampled_test["group_id"].to_numpy()
)

In [44]:
model = CatBoostClassifier(iterations=100, eval_metric="NDCG:top=10")
model.fit(resampled_train_pool, eval_set=resampled_test_pool, early_stopping_rounds=30, plot=True, verbose=False)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

In [45]:
print("Resampled requests")
metrics = model.eval_metrics(resampled_test_pool, ["AUC:type=Ranking", "NDCG", "NDCG:top=10"])
for metric in metrics:
    print(metric)
    print(np.mean(metrics[metric]))


print("-" * 50)
print("Original requests")
metrics = model.eval_metrics(test_pool, ["AUC:type=Ranking", "NDCG", "NDCG:top=10"])
for metric in metrics:
    print(metric)
    print(np.mean(metrics[metric]))

Resampled requests
AUC:type=Ranking
0.8828499552484331
NDCG:type=Base
0.8532124584282553
NDCG:top=10;type=Base
0.847666255476772
--------------------------------------------------
Original requests
AUC:type=Ranking
0.9289108938616663
NDCG:type=Base
0.983213578288152
NDCG:top=10;type=Base
0.9828333866597855


In [46]:
model = CatBoostRanker(iterations=100, eval_metric="NDCG:top=10")
model.fit(resampled_train_pool, eval_set=resampled_test_pool, early_stopping_rounds=30, plot=True, verbose=False)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

In [47]:
print("Resampled requests")
metrics = model.eval_metrics(resampled_test_pool, ["AUC:type=Ranking", "NDCG", "NDCG:top=10"])
for metric in metrics:
    print(metric)
    print(np.mean(metrics[metric]))

print("-" * 50)
print("Original requests")
metrics = model.eval_metrics(test_pool, ["AUC:type=Ranking", "NDCG", "NDCG:top=10"])
for metric in metrics:
    print(metric)
    print(np.mean(metrics[metric]))

Resampled requests
AUC:type=Ranking
0.8513167444832552
NDCG:type=Base
0.8574596983545525
NDCG:top=10;type=Base
0.8517356857653241
--------------------------------------------------
Original requests
AUC:type=Ranking
0.9122489214804981
NDCG:type=Base
0.9845768455072483
NDCG:top=10;type=Base
0.9842505969549287


### Ranking tricks: non-binary targets

In [48]:
# Non-binary target with purchases within 30 minutes of request
# purchase -> 2, cart update -> 1, view -> 0

# Build request timestamp per request_id
request_times = (
    resampled_dataset
    .group_by("request_id")
    .agg(
        pl.col("timestamp").min().alias("request_ts"),
        pl.col("user_id").first().alias("user_id"),
    )
)

# Load purchases from the original actions
purchases = (
    pl.read_parquet(ACTIONS_PATH)
    .filter(pl.col("action_type") == "AT_Purchase")
    .select(
        pl.col("user_id"),
        pl.col("item_id"),
        (pl.col("timestamp") * 1_000_000).cast(pl.Datetime).alias("purchase_ts"),
    )
)

# Attach request timestamps to each (request_id, item_id)
req_items = (
    resampled_dataset
    .select("request_id", "user_id", "item_id")
    .unique()
    .join(request_times, on=["request_id", "user_id"], how="inner")
    .sort("request_ts")
)

purchases = purchases.sort("purchase_ts")

# Use all purchases within 30 minutes after request_ts
req_items = (
    req_items
    .join(purchases, on=["user_id", "item_id"], how="left")
    .filter(
        (pl.col("purchase_ts") >= pl.col("request_ts")) &
        (pl.col("purchase_ts") <= (pl.col("request_ts") + pl.duration(minutes=30)))
    )
    .group_by(["request_id", "item_id"])
    .agg(pl.len().alias("purchases_30m"))
    .with_columns((pl.col("purchases_30m") > 0).alias("purchased_30m"))
    .select("request_id", "item_id", "purchased_30m")
)

resampled_dataset = resampled_dataset.join(req_items, on=["request_id", "item_id"], how="left")
resampled_dataset = resampled_dataset.with_columns(
    pl.col("purchased_30m").fill_null(False)
)


def build_target(action_type: str, purchased_30m: bool):
    if purchased_30m:
        return 2.0
    if action_type == "AT_CartUpdate":
        return 1.0
    return 0.0


In [49]:
resampled_cbm_dataset = (
    resampled_dataset
    .with_columns(
        pl.col("request_id").cast(str).alias("group_id"),
        pl.struct(["action_type", "purchased_30m"]).map_elements(lambda x: build_target(x["action_type"], x["purchased_30m"]), return_dtype=float).alias("target")
    )
    .with_columns(
        target=pl.col('target').max().over(partition_by=[pl.col('group_id'), pl.col('item_id')])
    )
    .unique()
    .sort("timestamp")
)

resampled_train, resampled_test = train_test_split(resampled_cbm_dataset, test_size=0.2, shuffle=False)

resampled_train = resampled_train.sort("group_id")
resampled_test = resampled_test.sort("group_id")

In [51]:
model = CatBoostRanker(iterations=100, eval_metric="NDCG:top=10")
model.fit(resampled_train_pool, eval_set=resampled_test_pool, early_stopping_rounds=30, plot=True, verbose=False)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

In [52]:
print("Original requests")
metrics = model.eval_metrics(test_pool, ["AUC:type=Ranking", "NDCG", "NDCG:top=10"])
for metric in metrics:
    print(metric)
    print(np.mean(metrics[metric]))

Original requests
AUC:type=Ranking
0.9122489214804981
NDCG:type=Base
0.9845768455072483
NDCG:top=10;type=Base
0.9842505969549287
